# Step 2 - Generate audio book with Azure Speech

In [ ]:
%pip install openai
%pip install python-dotenv
%pip install pydub
%pip install azure-core
%pip install azure-cognitiveservices-speech
%pip install azure-ai-documentintelligence

In [ ]:
# Add azure open ai endpoint
import os,re
from dotenv import load_dotenv

load_dotenv(dotenv_path='../keys.env')
azure_openai_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
azure_openai_key = os.getenv('AZURE_OPENAI_KEY')
azure_speech_key = os.getenv('AZURE_SPEECH_KEY')
azure_speech_region = os.getenv('AZURE_SPEECH_REGION')

folder_path = '../data/hachette'  # Adjust this path as needed

In [ ]:
def convert_markdown_to_ssml(input_file, output_file, language='en'):
    """
    Converts a markdown file to SSML format for Azure Speech Services.
    Handles titles separately and removes problematic line breaks.
    """
    
    # Set the voice based on the detected language (you might need to adjust voice names)

    if language == 'fr':
        voice = "fr-FR-Vivienne:DragonHDLatestNeural"  # Example French voice
    elif language == 'en':
        voice = "en-US-Ava3:DragonHDLatestNeural"  # Example English voice
    else:
        voice = "en-US-Andrew3:DragonHDLatestNeural"  # Default to English
    
    try:
        # Read the markdown file
        with open(input_file, 'r', encoding='utf-8') as f:
            content = f.read()
        

        # Remove PageBreaks
        lines = content.split('\n')
        i=0
        lines2 = list()
        # Loop through all the lines in the file
        while i < len(lines):
            line = lines[i].strip()
            if line == '<!-- PageBreak -->':
                # Remove the empty lines before and after
                if lines[i-1].strip() =='':
                    lines2.pop()
                if lines[i+1].strip() =='':
                    i=i+1
            else:
                lines2.append(escape(line))
            i += 1
        
        # tag titles
        lines = lines2.copy()
        i=0
        lines2.clear()
        while i < len(lines):
            line = lines[i].strip()
            if line.startswith('#'):
                title_level = len(re.match(r'^(#+)', line).group(1))
                title_text = line.replace('#', '').strip()
                # Replace the line with an empty string
                lines2.append(f'\n<s>{title_text}</s>\n')
          
            else:
                lines2.append(line)
            i += 1
            
        content = '\n'.join(lines2)
        
        # Clean the text - remove markdown formatting
        content = re.sub(r'\*\*(.*?)\*\*', r'\1', content)  # Remove bold
        content = re.sub(r'\*(.*?)\*', r'\1', content)      # Remove italic
        content = re.sub(r'!\[(.*?)\]\(.*?\)', '', content) # Remove images
        content = re.sub(r'\[(.*?)\]\(.*?\)', r'\1', content) # Convert links to text
   
         # Construct SSML
        ssml = f'''<speak version="1.0" xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="{language}">\n<voice name="{voice}">\n'''
          
        # Add paragraphs with appropriate breaks
        paragraphs = re.split(r'\n\s*\n', content.strip())
        for paragraph in paragraphs:
            if paragraph.strip():
                # Clean paragraph text
                cleaned_paragraph = paragraph.replace('\n', ' ').strip()
                if cleaned_paragraph:
                    # ssml += f'{cleaned_paragraph}\n'
                    ssml += f'<p>{cleaned_paragraph}</p>\n'
        
        ssml += '</voice>\n</speak>'
        
        # Write SSML to output file
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(ssml)
            
        return True
    except Exception as e:
        print(f"Error converting {input_file} to SSML: {e}")
        return False
    
def escape( str_xml: str ):
    str_xml = str_xml.replace("&", "&amp;")
    str_xml = str_xml.replace("<", "&lt;")
    str_xml = str_xml.replace(">", "&gt;")
    str_xml = str_xml.replace("\"", "&quot;")
    str_xml = str_xml.replace("'", "&apos;")
    return str_xml

def process_markdown_to_ssml(directory):
    """
    Process all markdown files in a directory and convert them to SSML files.
    """
    converted_files = []
    
    # Look for chapter directories from split_markdown_by_chapters function
    for root, dirs, files in os.walk(directory):
        for filename in files:
            if filename.endswith(".md"):
                md_file_path = os.path.join(root, filename)
                ssml_file_path = os.path.join(root, f"{os.path.splitext(filename)[0]}.ssml")
                
                print(f"Converting {md_file_path} to SSML...")
                if convert_markdown_to_ssml(md_file_path, ssml_file_path):
                    converted_files.append(ssml_file_path)
                    print(f"Successfully converted to {ssml_file_path}")
                else:
                    print(f"Failed to convert {md_file_path}")
    
    return converted_files

# Process markdown files in the root directory
converted_ssml_files = process_markdown_to_ssml(folder_path)
print(f"Converted {len(converted_ssml_files)} markdown files to SSML")

## Generate audio book 

In [ ]:
temp_folder = folder_path+"/temp_audio"  # Temporary folder for page-level audio
# Create temporary folder if it doesn't exist
if not os.path.exists(temp_folder):
    os.makedirs(temp_folder)

In [ ]:
# NEW CODE - This code use the BATCH MODE with REST API

import json
import os
import time
import uuid
from pathlib import Path
import requests

# Replace with your Azure Speech Services subscription key and region
azure_speech_key = os.environ['AZURE_SPEECH_KEY']
azure_speech_region = os.environ['AZURE_SPEECH_REGION']
azure_destination_container_url = os.environ['AZURE_DESTINATION_CONTAINER_URL']

API_VERSION = "2024-04-01"

def _create_job_id():
    # the job ID must be unique in current speech resource
    # you can use a GUID or a self-increasing number
    return uuid.uuid4()

def _authenticate():
    return {'Ocp-Apim-Subscription-Key': azure_speech_key}

def submit_synthesis(job_id: str, text : str) -> bool:
    url = f'https://{azure_speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = {
        'Content-Type': 'application/json'
    }
    header.update(_authenticate())

    payload = {
        "inputKind": "SSML", # or PlainText
        #'synthesisConfig': {
        #    "voice": "en-US-AvaMultilingualNeural",
        #},
        # Replace with your custom voice name and deployment ID if you want to use custom voice.
        # Multiple voices are supported, the mixture of custom voices and platform voices is allowed.
        # Invalid voice name or deployment ID will be rejected.
        #'customVoices': {
            # "YOUR_CUSTOM_VOICE_NAME": "YOUR_CUSTOM_VOICE_ID"
        #},
        "inputs": [
            {
                "content": text
            },
        ],
        "BatchSynthesisOutputs": {
            "result" :"result",
            "summary": "summary",
        },
        "properties": {
            # all properties doc https://learn.microsoft.com/en-us/rest/api/batchtexttospeech/batch-syntheses/create?view=rest-batchtexttospeech-2024-04-01&tabs=HTTP#batchsynthesisproperties

            # see https://learn.microsoft.com/en-us/azure/ai-services/speech-service/rest-text-to-speech?tabs=nonstreaming#audio-outputs
            "outputFormat": "riff-48khz-16bit-mono-pcm", # best quality
            "decompressOutputFiles": True,
            #"concatenateResult": False,
            "destinationContainerUrl": azure_destination_container_url
        },
    }

    response = requests.put(url, json.dumps(payload), headers=header)
    if response.status_code < 400:
        print('Batch synthesis job submitted successfully')
        print(f'Job ID: {response.json()["id"]}')
        return True
    else:
        print(f'Failed to submit batch synthesis job: [{response.status_code}], {response.text}')
        return False

def get_synthesis(job_id: str):
    url = f'https://{azure_speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        # print('Get batch synthesis job successfully')
        # print(response.json())
        return response.json()['status']
    else:
        print(f'Failed to get batch synthesis job: {response.text}')

def get_synthesis_audio_folder(job_id: str):
    url = f'https://{azure_speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses/{job_id}?api-version={API_VERSION}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        return response.json()['outputs']['result']
    else:
        print(f'Failed to get batch synthesis job: {response.text}')

def list_synthesis_jobs(skip: int = 0, max_page_size: int = 100):
    """List all batch synthesis jobs in the subscription"""
    url = f'https://{azure_speech_region}.api.cognitive.microsoft.com/texttospeech/batchsyntheses?api-version={API_VERSION}&skip={skip}&maxpagesize={max_page_size}'
    header = _authenticate()
    response = requests.get(url, headers=header)
    if response.status_code < 400:
        print(f'List batch synthesis jobs successfully, got {len(response.json()["values"])} jobs')
        print(response.json())
    else:
        print(f'Failed to list batch synthesis jobs: {response.text}')

def text_to_speech(job_id, text):
    """
    Converts text to speech using BATCH MODE (REST API)
    """
    i = 1
    try:
        if submit_synthesis(job_id, text):
            while True:
                status = get_synthesis(job_id)
                if status == 'Succeeded':
                    # print('batch synthesis job succeeded')
                    return True
                elif status == 'Failed':
                    print('batch synthesis job failed')
                    return False
                else:
                    print(f'Batch synthesis job is still running, status [{status}] ({i})')
                    i += 1
                    time.sleep(15)
        
    except Exception as e:
        print(f"Error during speech synthesis: {e}")
        return False

def process_chapter_audio(chapter_dir):
    """
    Processes each chapter file in the given directory to generate audio.
    """
    chapter_audio_folders = []
    for filename in os.listdir(chapter_dir):
        if filename == "0__.ssml":#filename.endswith(".ssml"):
            chapter_file_path = os.path.join(chapter_dir, filename)
            try:
                with open(chapter_file_path, "r", encoding="utf-8") as chapter_file:
                    chapter_content = chapter_file.read()

                # Generate audio from the chapter content
                # output_filename = f"{os.path.splitext(filename)[0]}.wav"
                # output_path = os.path.join(chapter_dir, output_filename)
                job_id = _create_job_id()
                if text_to_speech(job_id, chapter_content):
                    chapter_audio_folders.append(get_synthesis_audio_folder(job_id))
                else:
                    print(f"Failed to generate audio for {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    return chapter_audio_folders

chapter_audio_folders = []
for subdir, dirs, files in os.walk(folder_path):
        print(f"Reading directory: {subdir}")
        # process each chapt files 0__.ssml
        for filename in os.listdir(subdir):
            # Let's process only the first chapter for now
            if filename == "0__.ssml":
                print(f"Process audio from ssml file: {subdir}/{filename}")
                chapter_dir = subdir
                chapter_audio_folders.append(process_chapter_audio(chapter_dir))
                print (f"Audio file generated in folder {chapter_audio_folders[-1]} in storage container")



In [ ]:

def combine_audio_files(input_files, output_file):
    """
    Combines multiple audio files into a single audio file.
    """
    combined_audio = AudioSegment.empty()
    for file in input_files:
        audio = AudioSegment.from_wav(file)
        combined_audio += audio

    combined_audio.export(output_file, format="wav")
    print(f"Combined audio saved to {output_file}")

# Combine the audio files for all chapters
if chapter_audio_files:
    final_output_path = os.path.join(directory, f"{os.path.splitext(filename)[0]}_combined_chapters.wav")
    combine_audio_files(chapter_audio_files, final_output_path)

    # Clean up temporary audio files
    for file in chapter_audio_files:
        os.remove(file)
    print(f"Temporary audio files removed from {temp_folder}")
else:
    print("No audio files were created for this PDF.")